### Load and split the data

In [ ]:
import os, random, hashlib, time
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from imutils import paths
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

path = "../data/processed"
IMG_SIZE = (224, 224)
SEED = 42

image_paths = list(paths.list_images(path))

# remove broken imgs and duplicates (hash)
hashes = {}
for p in image_paths:
    if cv2.imread(p) is None:
        continue
    with open(p, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()
    hashes.setdefault(h, []).append(p)

files = []
for g in hashes.values():
    if len({p.split(os.path.sep)[-2] for p in g}) == 1:
        files.append(g[0])

random.seed(SEED)
random.shuffle(files)

# read, resize, label
X, labels = [], []
for p in files:
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    X.append(cv2.resize(img, IMG_SIZE))
    labels.append(p.split(os.path.sep)[-2])

CLASSES = sorted(set(labels))
X = np.array(X)
y = np.array([CLASSES.index(l) for l in labels])

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

def make_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(lambda a, b: (tf.cast(a, tf.float32) / 255.0, b))  # normalize
    ds = ds.cache()
    if training:
        ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, training=True)
val_ds = make_ds(X_val, y_val)
test_ds = make_ds(X_test, y_test)

print(X_train.shape, X_val.shape, X_test.shape)  # sizes of the 3 sets
print(CLASSES)  # class names

### Build the CNN and inspect it

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),

    # augmentation, only active when training
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.05, 0.05),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASSES), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

### Class weights computed on the training set only

In [ ]:
weights = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_train)
class_weight = dict(enumerate(weights))
print(class_weight)  # bigger weight = rarer class

### Callbacks: save the best model, stop early

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_model.keras", monitor="val_accuracy", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
]

### Train the model and measure the training time

In [ ]:
EPOCHS = 10  # put 1 for a quick test

start = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)
duration = time.time() - start
print(round(duration / 60, 1))  # training time in min

### Learning curves: accuracy and loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()